# 09 — Contest week: do LLM news signals improve the RL agent?  *(branch `exp/09-contest-news`)*

A second, independent test of the ablation in 05, on the FinAI Contest 2025 Task 1 data: the contest's
1-second limit-order book for **2021-04-07 → 04-13** and the organisers' **DeepSeek scores of 279 Bitcoin
news articles** of the same week (HF `SecureFinAI-Lab/FinRL_BTC_news_signals`). Unlike the daily tweet
aggregate, these signals are **intraday** (a new score with every article, median ~20 min apart).

* signals rebuilt **causally from publication times** (`rltrade/contest.py`): a bar sees only articles
  available before it ends; an article waits for its Yahoo URL time when that is later
* **same pipeline, only the data changes**: Alpha101 → the GRU selected in 04 (applied, not retrained) →
  DDQN with the hyperparameters tuned in 06; selection on validation, ≥ 5 validation trades; test once
* states A–D at realistic costs (0.10 % + 0.02 %), plus A and D at 0.02 % per trade; **10 seeds** each

**Caveat — read the result accordingly:** 5.7 days in total; validation and test are ~20 hours each. Seeds
measure training noise on ONE short test window, not market variation, so p-values are indicative only.

Inputs (upload once to `thesis_rl/contest/data/`; never committed): `contest_1min_news.csv`, built locally with
`python -m rltrade.contest build BTC_1sec.csv news_train.csv contest_1min_news.csv` — or put `BTC_1sec.csv` and
`news_train.csv` there and the next cell builds it.

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/09-contest-news"
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available(): print('WARNING: no GPU - Runtime > Change runtime type > T4 GPU')

In [ ]:
seeding.record_environment(L.report / "environment", "09_contest_news")

## Data, factors, hyperparameters

In [ ]:
import json, shutil
from pathlib import Path
from rltrade import contest
from rltrade.prepare import prepare
from rltrade.rnn.train import export_factors
C = contest.contest_layout(Path(DRIVE_ROOT) / "contest")
for p in C.values():
    p.mkdir(parents=True, exist_ok=True) if p.suffix == "" else p.parent.mkdir(parents=True, exist_ok=True)
if not C["data"].exists():
    contest.build_dataset(C["data"].parent / "BTC_1sec.csv", C["data"].parent / "news_train.csv", C["data"])
if not (C["prepared"] / "meta.json").exists():
    prepare(C["data"], C["prepared"], signal_lag="none")          # signals are already causal per bar
if not (C["factors"] / "factors_test.npy").exists():
    sel = json.loads((L.rnn_tuning / "selected.json").read_text())   # the GRU selected in 04, applied unchanged
    export_factors(L.rnn_tuning / "final" / f"{sel['arch']}_s{sel['seed']}", C["prepared"], C["factors"])
HP = L.runs / "tuning" / "best_hp_mt5.json"                         # DDQN hyperparameters tuned in 06
print(json.loads(HP.read_text())["ddqn"])

## Runs

In [ ]:
# ~15 min per DDQN run on a T4 / CPU; 60 runs + 4 baselines. Finished runs are skipped on a re-run.
contest.run_groups(Path(DRIVE_ROOT) / "contest", HP)

## Results

In [ ]:
from rltrade import contest_report
print(contest_report.report(Path(DRIVE_ROOT) / "contest"))